In [10]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error
from xgboost import XGBRegressor

# ==============================
# 2. LOAD DATA
# ==============================
df = pd.read_csv("QCFE1_Perfect_Demand.csv")

# Convert to datetime
df['Order_date'] = pd.to_datetime(df['Order_date'])

# Sort (VERY IMPORTANT for time series)
df = df.sort_values(by=['City', 'Product_Category', 'Order_date'])

# ==============================
# 3. AGGREGATE DATA
# ==============================
df = df.groupby(['City', 'Product_Category', 'Order_date'])['Items_Count'].sum().reset_index()

# ==============================
# 4. DATE FEATURES
# ==============================
df['Day'] = df['Order_date'].dt.day
df['Month'] = df['Order_date'].dt.month
df['Day_of_week'] = df['Order_date'].dt.dayofweek
df['Week_of_Year'] = df['Order_date'].dt.isocalendar().week.astype(int)

df['Is_Weekend'] = (df['Day_of_week'] >= 5).astype(int)
df['Season'] = (df['Month'] % 12 // 3).astype(int)

# ==============================
# 5. FESTIVAL FEATURE
# ==============================
festival_days = pd.to_datetime([
    "2025-08-09",
    "2025-09-22","2025-09-23","2025-09-24",
    "2025-09-25","2025-09-26","2025-09-27",
    "2025-09-28","2025-09-29","2025-09-30",
    "2025-10-02","2025-10-20"
])

df['Is_Festival'] = df['Order_date'].isin(festival_days).astype(int)

# ==============================
# 6. LAG FEATURES (STRONG)
# ==============================
for lag in [1, 2, 3, 7, 14]:
    df[f'lag_{lag}'] = df.groupby(['City', 'Product_Category'])['Items_Count'].shift(lag)

# ==============================
# 7. ROLLING FEATURES
# ==============================
df['rolling_mean_3'] = df.groupby(['City', 'Product_Category'])['Items_Count']\
    .transform(lambda x: x.shift(1).rolling(3).mean())

df['rolling_mean_7'] = df.groupby(['City', 'Product_Category'])['Items_Count']\
    .transform(lambda x: x.shift(1).rolling(7).mean())

df['rolling_std_7'] = df.groupby(['City', 'Product_Category'])['Items_Count']\
    .transform(lambda x: x.shift(1).rolling(7).std())

# ==============================
# 8. DROP NaN (due to lags)
# ==============================
df = df.dropna().reset_index(drop=True)

# ==============================
# 9. ONE-HOT ENCODING (FIXED)
# ==============================
df = pd.get_dummies(df, columns=['City', 'Product_Category'])

# ==============================
# 10. FEATURES & TARGET
# ==============================
features = [col for col in df.columns if col not in ['Items_Count', 'Order_date']]

X = df[features]

# LOG TRANSFORM (VERY IMPORTANT)
y = np.log1p(df['Items_Count'])

# ==============================
# 11. TIME-BASED SPLIT
# ==============================
split = int(len(df) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# ==============================
# 12. MODEL (TUNED)
# ==============================
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

# ==============================
# 13. PREDICTION
# ==============================
y_pred_log = model.predict(X_test)

# Convert back from log
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

# ==============================
# 14. METRICS
# ==============================
mae = mean_absolute_error(y_test_actual, y_pred)
r2 = r2_score(y_test_actual, y_pred)
mape = mean_absolute_percentage_error(y_test_actual, y_pred)

print("MAE:", mae)
print("R2 Score:", r2)
print("MAPE:", mape)
print("MAPE %:", mape * 100)



MAE: 141.55874904184557
R2 Score: 0.9487350649131152
MAPE: 0.19918810127414896
MAPE %: 19.918810127414897


In [11]:
joblib.dump(model, "model.pkl")
joblib.dump(features, "features.pkl")
joblib.dump(df, "processed_df.pkl")  # VERY IMPORTANT for prediction

['processed_df.pkl']

In [12]:
model = joblib.load("model.pkl")
features = joblib.load("features.pkl")
df = joblib.load("processed_df.pkl")

In [13]:
def predict_demand(city, category, date):

    date = pd.to_datetime(date)

    temp = df[(df['City_'+city] == 1) & (df['Product_Category_'+category] == 1)]
    temp = temp.sort_values(by='Order_date')

    last = temp[temp['Order_date'] < date].tail(14)

    if len(last) < 7:
        return "Not enough data"

    row = {}

    row['Day'] = date.day
    row['Month'] = date.month
    row['Day_of_week'] = date.dayofweek
    row['Week_of_Year'] = date.isocalendar().week
    row['Is_Weekend'] = 1 if date.dayofweek >= 5 else 0
    row['Season'] = (date.month % 12 // 3)

    row['lag_1'] = last['Items_Count'].iloc[-1]
    row['lag_2'] = last['Items_Count'].iloc[-2]
    row['lag_3'] = last['Items_Count'].iloc[-3]
    row['lag_7'] = last['Items_Count'].iloc[-7]
    row['lag_14'] = last['Items_Count'].iloc[-14]

    row['rolling_mean_7'] = last['Items_Count'].tail(7).mean()

    row_df = pd.DataFrame([row])

    # Add missing columns
    for col in features:
        if col not in row_df.columns:
            row_df[col] = 0

    # Set correct city/category
    row_df['City_'+city] = 1
    row_df['Product_Category_'+category] = 1

    row_df = row_df[features]

    pred = np.expm1(model.predict(row_df))

    return int(pred[0])

In [14]:
mae = mean_absolute_error(y_test_actual, y_pred)
r2 = r2_score(y_test_actual, y_pred)
mape = mean_absolute_percentage_error(y_test_actual, y_pred)

print("MAE:", mae)
print("R2 Score:", r2)
print("MAPE:", mape)
print("MAPE %:", mape * 100)


MAE: 141.55874904184557
R2 Score: 0.9487350649131152
MAPE: 0.19918810127414896
MAPE %: 19.918810127414897
